In [17]:
import numpy as np 
import pandas as pd 
from darts.models import Prophet
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller
from darts.dataprocessing.pipeline import Pipeline
from darts import TimeSeries
from darts.timeseries import concatenate
from darts.models.forecasting.regression_ensemble_model import RegressionEnsembleModel
from darts.models import XGBModel,RandomForest,LightGBMModel,CatBoostModel
from sklearn.ensemble import BaggingRegressor
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from utils import pad_series_pair



In [18]:
DATA_ROOT ='data/'
df = pd.read_csv(DATA_ROOT + 'multimodal_data.csv')


In [19]:
feed_comp_mapping = {group: idx+1 for idx, group in enumerate(df['group'].unique())}

df['feed_comp_num'] = df['group'].map(feed_comp_mapping)

dummy_df = pd.get_dummies(df, columns=['group'], prefix='group')

group_counts = df['group'].value_counts()
total_samples = len(df)
feed_comp_percentages = (group_counts / total_samples * 100).round(2)

print("Feed Composition Distribution:")
for group, percentage in feed_comp_percentages.items():
    print(f"{group} (#{feed_comp_mapping[group]}): {percentage}%")

dummy_df.shape
df['feed_comp_num'] = dummy_df['feed_comp_num']

Feed Composition Distribution:
FBC + PPI + SPC (#1): 33.33%
FBC + SPI + SPC (#2): 33.33%
FBC + Gluten + SPC (#3): 33.33%


In [20]:
desired_features = ['Time(min)','Bulk Density','Extruder Load',
    'Extruder RPM', 'Extruder Water', 'Temp Zone 3', 'Temp Zone 4',
    'Die Temp','SME-Watt','Score','Score_unsup_CV','Score_semi_sup_CV','ts','group','feed_comp_num']
df = df[desired_features]
df['Time-ts'] = np.arange(len(df))


In [21]:
target_cols = ['Die Temp','SME-Watt','Score']
past_covariates = ['Temp Zone 3','Temp Zone 4','Bulk Density','Extruder Load', 'Extruder Water','feed_comp_num']


In [22]:
grouped_ts = df.groupby('group')

past_covariates_train_list = []
past_covariates_test_list = []
target_train_list = []
target_test_list = []
test_df = pd.DataFrame()
train_df = pd.DataFrame()

for name, group in grouped_ts:
    
    group['Time(min)-group-all'] = np.arange(len(group))
    total_len = len(group)
    train_idx = int(0.7 * total_len)

    train_data = group.iloc[:train_idx].copy()
    test_data = group.iloc[train_idx:].copy()

    train_df = pd.concat([train_df, train_data], ignore_index=True)
    test_df = pd.concat([test_df, test_data], ignore_index=True)

    train_target = TimeSeries.from_dataframe(train_data, time_col='Time(min)-group-all', value_cols=target_cols)
    test_target = TimeSeries.from_dataframe(test_data, time_col='Time(min)-group-all', value_cols=target_cols)

    train_past_covariates = TimeSeries.from_dataframe(train_data, time_col='Time(min)-group-all', value_cols=past_covariates)
    test_past_covariates = TimeSeries.from_dataframe(test_data, time_col='Time(min)-group-all', value_cols=past_covariates)

    target_train_list.append(train_target)
    target_test_list.append(test_target)

    past_covariates_train_list.append(train_past_covariates)
    past_covariates_test_list.append(test_past_covariates)


In [23]:
train_df['split'] = 'train'
test_df['split'] = 'test'


In [24]:
target_train_series = concatenate(target_train_list,ignore_time_axis=True)
target_test_series = concatenate(target_test_list,ignore_time_axis=True)

past_covariates_train_series = concatenate(past_covariates_train_list,ignore_time_axis=True,axis=0)
past_covariates_test_series = concatenate(past_covariates_test_list,ignore_time_axis=True,axis=0)



In [25]:
print("Train set length:", len(train_df))
print("Test set length:", len(test_df))

Train set length: 63
Test set length: 27


In [26]:
target_scaler = Scaler(scaler=MinMaxScaler(feature_range=(-1,1)))
past_covariates_scaler = Scaler(scaler=MinMaxScaler(feature_range=(-1,1)))

n_lags = 2 # Same as model's lags parameter
current_length = len(target_test_series)
# Calculate the next number that's divisible by n_lags
required_length = ((current_length + n_lags - 1) // n_lags) * n_lags + n_lags

target_train_series_scaled = target_scaler.fit_transform(target_train_series)
target_test_series_scaled = target_scaler.transform(target_test_series)

past_covariates_train_series_scaled = past_covariates_scaler.fit_transform(past_covariates_train_series)
past_covariates_test_series_scaled = past_covariates_scaler.transform(past_covariates_test_series)


In [27]:
XGB_model = XGBModel(
    lags=n_lags,
    n_estimators=30,
    lags_past_covariates=n_lags,
    output_chunk_length=1,
    add_encoders=None,
    learning_rate=0.1,
    max_depth=2, 
    min_child_weight=1,
    subsample=1,  
    colsample_bytree=1.0
)

XGB_model.fit(target_train_series_scaled,past_covariates=past_covariates_train_series_scaled)



XGBModel(lags=2, lags_past_covariates=2, lags_future_covariates=None, output_chunk_length=1, output_chunk_shift=0, add_encoders=None, likelihood=None, quantiles=None, random_state=None, multi_models=True, use_static_covariates=True, n_estimators=30, learning_rate=0.1, max_depth=2, min_child_weight=1, subsample=1, colsample_bytree=1.0)

In [28]:
GBM_model = LightGBMModel(
    lags=n_lags,
    n_estimators=30,
    lags_past_covariates=n_lags,
    output_chunk_length=1,
    add_encoders=None,
    learning_rate=0.1,
    max_depth=2,
    num_leaves=4,
    min_child_samples=1,
    subsample=1,
    colsample_bytree=1.0
)

# Fit the model on the training data
GBM_model.fit(target_train_series_scaled,past_covariates=past_covariates_train_series_scaled)



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000405 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 300
[LightGBM] [Info] Number of data points in the train set: 61, number of used features: 18
[LightGBM] [Info] Start training from score 0.216820
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000365 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 300
[LightGBM] [Info] Number of data points in the train set: 61, number of used features: 18
[LightGBM] [Info] Start training from score 0.147097
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000355 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 300
[LightGBM] [Info] Number of data points in the train set: 61, number of used features: 18
[LightGBM] [Info] Start training from s

LightGBMModel(lags=2, lags_past_covariates=2, lags_future_covariates=None, output_chunk_length=1, output_chunk_shift=0, add_encoders=None, likelihood=None, quantiles=None, random_state=None, multi_models=True, use_static_covariates=True, categorical_past_covariates=None, categorical_future_covariates=None, categorical_static_covariates=None, n_estimators=30, learning_rate=0.1, max_depth=2, num_leaves=4, min_child_samples=1, subsample=1, colsample_bytree=1.0)

In [29]:
test_group = train_df.groupby('group')

# XGBoost
train_df_xgb = pd.DataFrame()
for i, (name, group) in enumerate(test_group):
    idx = group['Time(min)-group-all'].values
    ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=target_cols)
    scaled_ts = target_scaler.transform(ts)
    past_covariates_ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=past_covariates)
    scaled_past_covariates_ts = past_covariates_scaler.transform(past_covariates_ts)
    target_test_series_pred_xgb = XGB_model.historical_forecasts(series=scaled_ts,past_covariates=scaled_past_covariates_ts,start=1,forecast_horizon=1,retrain=False)
    target_test_series_pred_xgb = target_scaler.inverse_transform(target_test_series_pred_xgb)
    pred_df_xgb = target_test_series_pred_xgb.pd_dataframe()
    
    pred_df_xgb.rename(columns={col: f'{col}_xgb' for col in pred_df_xgb.columns}, inplace=True)
    train_g_df = group.copy()

    for col in target_cols:
        train_g_df[f'{col}_xgb'] = np.nan
        train_g_df[f'{col}_xgb'].iloc[0:n_lags] = train_g_df[col].iloc[0:n_lags].values
        train_g_df[f'{col}_xgb'].iloc[n_lags:] = pred_df_xgb[ f'{col}_xgb'].values
    train_df_xgb = pd.concat([train_df_xgb, train_g_df])


# LightGBM
train_df_gbm = pd.DataFrame()
for i, (name, group) in enumerate(test_group):
    idx = group['Time(min)-group-all'].values
    ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=target_cols)
    scaled_ts = target_scaler.transform(ts)
    past_covariates_ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=past_covariates)
    scaled_past_covariates_ts = past_covariates_scaler.transform(past_covariates_ts)
    target_test_series_pred_gbm = GBM_model.historical_forecasts(series=scaled_ts,past_covariates=scaled_past_covariates_ts,start=int(idx[0]),forecast_horizon=1,retrain=False)
    target_test_series_pred_gbm = target_scaler.inverse_transform(target_test_series_pred_gbm)
    pred_df_gbm = target_test_series_pred_gbm.pd_dataframe()
    pred_df_gbm.rename(columns={col: f'{col}_gbm' for col in pred_df_gbm.columns}, inplace=True)
    train_g_df = group.copy()
    for col in target_cols:
        train_g_df[f'{col}_gbm'] = np.nan
        train_g_df[f'{col}_gbm'].iloc[0:n_lags] = train_g_df[col].iloc[0:n_lags].values
        train_g_df[f'{col}_gbm'].iloc[n_lags:] = pred_df_gbm[ f'{col}_gbm'].values
    train_df_gbm = pd.concat([train_df_gbm, train_g_df])



`start` value `1` corresponding to timestamp `1` is before the first predictable/trainable historical forecasting point for series at index: 0. Ignoring `start` for this series and beginning at first trainable/predictable time: 2. To hide these warnings, set `show_warnings=False`.
C:\Users\amirbg\AppData\Local\Temp\ipykernel_47152\2756225215.py:20: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the or

In [30]:
# Get the last 2 rows from each validation group and first row from each test group
val_last_rows = []
test_first_rows = []

val_group = train_df.groupby('group')
test_group = test_df.groupby('group')

lagged_test_df = pd.DataFrame()
for val_name, val in val_group:
    # Get last 2 rows from validation group
    for test_name, test in test_group:
        if val_name == test_name:
            # Get last 2 rows from validation group
            last_two_rows = val.tail(1)
            test = pd.concat([last_two_rows, test])
            lagged_test_df = pd.concat([lagged_test_df, test])


In [31]:
test_group = lagged_test_df.groupby('group')

# XGBoost
test_df_xgb = pd.DataFrame()
for i, (name, group) in enumerate(test_group):
    idx = group['Time(min)-group-all'].values
    ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=target_cols)
    scaled_ts = target_scaler.transform(ts)
    past_covariates_ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=past_covariates)
    scaled_past_covariates_ts = past_covariates_scaler.transform(past_covariates_ts)
    target_test_series_pred_xgb = XGB_model.historical_forecasts(series=scaled_ts,past_covariates=scaled_past_covariates_ts,start=int(idx[1]),forecast_horizon=1,retrain=False)
    target_test_series_pred_xgb = target_scaler.inverse_transform(target_test_series_pred_xgb)
    pred_df_xgb = target_test_series_pred_xgb.pd_dataframe()
    pred_df_xgb.rename(columns={col: f'{col}_xgb' for col in pred_df_xgb.columns}, inplace=True)
    test_g_df = group[group['split'] == 'test']
    for col in target_cols:
        test_g_df[f'{col}_xgb'] = np.nan
        test_g_df[f'{col}_xgb'].iloc[0] = test_g_df[col].iloc[0]
        test_g_df[f'{col}_xgb'].iloc[1:] = pred_df_xgb[ f'{col}_xgb'].values
    test_df_xgb = pd.concat([test_df_xgb, test_g_df])


# LightGBM
test_df_gbm = pd.DataFrame()
for i, (name, group) in enumerate(test_group):
    idx = group['Time(min)-group-all'].values
    ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=target_cols)
    scaled_ts = target_scaler.transform(ts)
    past_covariates_ts = TimeSeries.from_dataframe(group, time_col='Time(min)-group-all', value_cols=past_covariates)
    scaled_past_covariates_ts = past_covariates_scaler.transform(past_covariates_ts)
    target_test_series_pred_gbm = GBM_model.historical_forecasts(series=scaled_ts,past_covariates=scaled_past_covariates_ts,start=int(idx[1]),forecast_horizon=1,retrain=False)
    target_test_series_pred_gbm = target_scaler.inverse_transform(target_test_series_pred_gbm)
    pred_df_gbm = target_test_series_pred_gbm.pd_dataframe()
    pred_df_gbm.rename(columns={col: f'{col}_gbm' for col in pred_df_gbm.columns}, inplace=True)
    test_g_df = group[group['split'] == 'test']
    for col in target_cols:
        test_g_df[f'{col}_gbm'] = np.nan
        test_g_df[f'{col}_gbm'].iloc[0] = test_g_df[col].iloc[0]
        test_g_df[f'{col}_gbm'].iloc[1:] = pred_df_gbm[ f'{col}_gbm'].values
    test_df_gbm = pd.concat([test_df_gbm, test_g_df])




`start` value `21` corresponding to timestamp `21` is before the first predictable/trainable historical forecasting point for series at index: 0. Ignoring `start` for this series and beginning at first trainable/predictable time: 22. To hide these warnings, set `show_warnings=False`.
C:\Users\amirbg\AppData\Local\Temp\ipykernel_47152\798756417.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_g_df[f'{col}_xgb'] = np.nan
C:\Users\amirbg\AppData\Local\Temp\ipykernel_47152\798756417.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this 

In [32]:
pd.concat([train_df_xgb,test_df_xgb],axis=0).to_csv(
    'Simulations/Multi-modal-xgb-g.csv'
)

pd.concat([train_df_gbm,test_df_gbm],axis=0).to_csv(
    'Simulations/Multi-modal-gbm-g.csv'
)
